# 8. Option Pricing: Monte Carlo vs Black-Scholes

- **Objective**: Price both Calls and Puts using Monte Carlo, compare to exact formulas, and verify Put-Call Parity.

In [1]:
import numpy as np
import scipy.stats as stats

def mc_pricing(S0, K, r, sigma, T, N):
    Z = np.random.standard_normal(N)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    call_payoffs = np.maximum(ST - K, 0)
    put_payoffs = np.maximum(K - ST, 0)

    call_price = np.mean(call_payoffs) * np.exp(-r * T)
    put_price = np.mean(put_payoffs) * np.exp(-r * T)

    return call_price, put_price

def black_scholes(S0, K, r, sigma, T):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    call_price = S0 * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    put_price = K * np.exp(-r * T) * stats.norm.cdf(-d2) - S0 * stats.norm.cdf(-d1)

    return call_price, put_price

S0 = 100
K = 105
r = 0.05
sigma = 0.2
T = 1.0
N = 1000000

mc_call, mc_put = mc_pricing(S0, K, r, sigma, T, N)
bs_call, bs_put = black_scholes(S0, K, r, sigma, T)

print(f'Call Option: MC Estimate = ${mc_call:.4f} | BS Exact = ${bs_call:.4f}')
print(f'Put Option:  MC Estimate = ${mc_put:.4f} | BS Exact = ${bs_put:.4f}')

Call Option: MC Estimate = $8.0315 | BS Exact = $8.0214
Put Option:  MC Estimate = $7.9089 | BS Exact = $7.9004


## Put-Call Parity

- **Law of Arbitrage**: $C - P = S_0 - K e^{-rT}$
- Let's verify that the prices generated by our randomized Monte Carlo simulation actually obey this fundamental financial law.

In [2]:
left_side = mc_call - mc_put
right_side = S0 - K * np.exp(-r * T)

print(f'Put-Call Parity (Left Side: C - P):        ${left_side:.4f}')
print(f'Put-Call Parity (Right Side: S0 - Ke^-rT): ${right_side:.4f}')
print(f'Discrepancy (Arbitrage Error):             ${abs(left_side - right_side):.6f}')

Put-Call Parity (Left Side: C - P):        $0.1226
Put-Call Parity (Right Side: S0 - Ke^-rT): $0.1209
Discrepancy (Arbitrage Error):             $0.001738


## Questions for Understanding

- The discrepancy in Put-Call Parity is close to zero, but not exactly zero. Why?
- If the left side ($C-P$) was significantly higher than the right side in the real market, how could a trader make risk-free money? (Hint: Buy the cheap side, sell the expensive side).